In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS Silver;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_employee = spark.table("transactional_data.employee")

In [0]:
silver_employee = (
    df_employee
    .dropDuplicates()
    .withColumn("Employee_ID", upper(trim(col("Employee_ID"))))
    .withColumn("Employee_Name", initcap(trim(col("Employee_Name"))))
    .withColumn("Designation", initcap(trim(col("Designation"))))
    .withColumn("Manager_ID", upper(trim(col("Manager_ID"))))
    .withColumn("Manager_Name", initcap(trim(col("Manager_Name"))))
    .withColumn("Monthly_Salary", col("Monthly_Salary").cast("decimal(10,2)"))
    .withColumn("Rating", col("Rating").cast("decimal(3,2)"))
)

In [0]:
silver_employee = silver_employee.withColumn(
    "Employee_Type",
    when(col("Designation").contains("Manager"), "Management")
    .when(col("Designation").contains("Chef"), "Kitchen")
    .when(col("Designation").contains("Waiter"), "Service")
    .when(col("Designation").contains("Cashier"), "Cash")
    .otherwise("Others")
)

In [0]:
silver_employee = silver_employee.withColumn(
    "Salary_Band",
    when(col("Monthly_Salary") >= 70000, "High")
    .when(col("Monthly_Salary") >= 40000, "Medium")
    .otherwise("Low")
)

In [0]:
silver_employee.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("silver.employee")

In [0]:
%sql
select * from silver.employee;

Employee_ID,Employee_Name,Designation,Manager_ID,Manager_Name,Monthly_Salary,Rating,Employee_Type,Salary_Band
EMP004,Priya Kapoor,Chef,EMP003,Amit Singh,45000.00,4.60,Kitchen,Medium
EMP009,Karan Malhotra,Delivery Coordinator,EMP002,Neha Verma,35000.00,4.60,Others,Low
EMP010,Riya Saxena,Cashier,EMP002,Neha Verma,30000.00,4.50,Cash,Low
EMP006,Anjali Gupta,Senior Waiter,EMP002,Neha Verma,32000.00,4.70,Service,Low
EMP007,Vikas Yadav,Waiter,EMP006,Anjali Gupta,28000.00,4.50,Service,Low
EMP003,Amit Singh,Head Chef,EMP001,Rahul Sharma,70000.00,4.80,Kitchen,High
EMP002,Neha Verma,Assistant Manager,EMP001,Rahul Sharma,65000.00,4.70,Management,Medium
EMP008,Sneha Tiwari,Waiter,EMP006,Anjali Gupta,29000.00,4.80,Service,Low
EMP005,Rohit Mishra,Chef,EMP003,Amit Singh,43000.00,4.40,Kitchen,Medium
EMP001,Rahul Sharma,Restaurant Manager,null,null,85000.00,4.90,Management,High


In [0]:
orders = spark.table("transactional_data.stg_restaurant_orders")

In [0]:
orders = (

orders

.dropDuplicates()

.withColumn("Restaurant_Name",initcap(trim(col("Restaurant_Name"))))

.withColumn("Subzone",initcap(trim(col("Subzone"))))

.withColumn("City",initcap(trim(col("City"))))

.withColumn("Item_Name",initcap(trim(col("Item_Name"))))

.withColumn("Customer_ID",upper(trim(col("Customer_ID"))))

.withColumn("Waiter_ID",upper(trim(col("Waiter_ID"))))

.withColumn("Chef_ID",upper(trim(col("Chef_ID"))))

)

In [0]:
orders = (

orders

.withColumn("Order_Date",to_date(col("ORDER_PLACED_AT")))

.withColumn("Order_Year",year(col("ORDER_PLACED_AT")))

.withColumn("Order_Month",month(col("ORDER_PLACED_AT")))

.withColumn("Month_Name",date_format(col("ORDER_PLACED_AT"),"MMMM"))

.withColumn("Day_Name",date_format(col("ORDER_PLACED_AT"),"EEEE"))

.withColumn("Quarter",quarter(col("ORDER_PLACED_AT")))

)

In [0]:
orders = orders.withColumn(

"Meal_Type",

when(hour("ORDER_PLACED_AT").between(6,11),"Breakfast")

.when(hour("ORDER_PLACED_AT").between(12,16),"Lunch")

.when(hour("ORDER_PLACED_AT").between(17,22),"Dinner")

.otherwise("Late Night")

)

In [0]:
orders = orders.withColumn(

"Weekend_Flag",

when(dayofweek("ORDER_PLACED_AT").isin([1,7]),"Yes")

.otherwise("No")

)

In [0]:
orders.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("silver.restaurant_orders")